**QUERY PIPELINE**

- User prompt
- Retrieve data from Milvus vector database
- Rank retrieved chunks
- Parse context + prompt to LLM
- Response

In [1]:
#import libraries

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

# from llama_index.llms.openai import OpenAI
from pymilvus import MilvusClient

load_dotenv()

True

In [2]:
#connect to milvus database

collection_name = "paper_chunks"

client = MilvusClient(
    uri="http://localhost:19530",
    token="root:Milvus",
)

In [3]:
#user prompt

user_prompt = "What are the parameters of SARIMA they have used?"

In [4]:
#retrieve data - embed the prompt and search the vector database

embed_model = OpenAIEmbedding()
query_embedding = embed_model.get_query_embedding(user_prompt)

search_results = client.search(
    collection_name=collection_name,
    data=[query_embedding],
    limit=10,
    output_fields=["title", "page", "chunk_index", "text"],
)

retrieved_chunks = search_results[0]
len(retrieved_chunks)

10

In [5]:
#ranking - sort by similarity score and keep the top chunks

top_k = 3

ranked_chunks = sorted(retrieved_chunks, key=lambda hit: hit["distance"], reverse=True)[:top_k]

for rank, hit in enumerate(ranked_chunks, start=1):
    print(f"rank {rank} | score: {hit['distance']:.4f} | page: {hit['entity']['page']}")

rank 1 | score: 0.8710 | page: 11
rank 2 | score: 0.8641 | page: 9
rank 3 | score: 0.8638 | page: 11


In [6]:
#parsing to llm - build context from ranked chunks and prompt the llm

context_str = "\n\n".join(hit["entity"]["text"] for hit in ranked_chunks)

prompt = f"""Answer the question using only the context below. If the answer isn't in the context, say you don't know.

Context:
{context_str}

Question: {user_prompt}
Answer:"""

llm = ChatOpenAI(model="gpt-4o-mini")

In [7]:
#response

response = llm.invoke(prompt)
print(response.text)

The parameters of SARIMA used are p, d, q, P, D, Q, and s (which is set to a constant value of 12).
